# Session 7 — Text classification at scale, with validation\n\nThe full loop, on our synthetic survey: **codebook → batch classify → validate against a human → agreement metrics → error analysis.** This notebook is deliberately the skeleton of Assignment 4, Option 1.

In [ ]:
import sys, pathlib
sys.path.append(str(pathlib.Path("..").resolve()))   # notebooks/ -> repo root
import course_utils

try:
    client = course_utils.get_client(verbose=True)
except RuntimeError:
    # First time here without check_setup? We can store the key now too:
    import getpass, os
    key = getpass.getpass("Paste your API key (input stays hidden): ")
    course_utils._persist("GEMINI_API_KEY", key)
    os.environ["GEMINI_API_KEY"] = key
    client = course_utils.get_client(verbose=True)

MODEL = course_utils.MODEL
print("Client ready.")

## 1. Load the fixed sample and the codebook

In [ ]:
import pandas as pd, json
sample = pd.read_csv("../data/sample_200.csv")
codebook = open("../data/codebook_concerns.md", encoding="utf-8").read()
print(len(sample), "texts"); sample.head(3)

## 2. Classify in batches of 25\n\nWatch the pattern: numbered batch in → JSON list out → collected into one table. (In-session we do the first 100 rows; scaling to all 200 is your take-home.)

In [ ]:
def classify_batch(texts, start_n):
    numbered = "\n".join(f"{start_n+i}. {t}" for i, t in enumerate(texts))
    r = client.models.generate_content(
        model=MODEL,
        contents=f"""Using ONLY the codebook below, assign exactly one category
(JOB/SURV/SKILL/FAIR/REL) to each numbered text. If none fits, use UNCLEAR.
Return a JSON list of objects with keys: n, category.
CODEBOOK:\n{codebook}\n\nTEXTS:\n{numbered}""",
        config={"temperature": 0, "response_mime_type": "application/json"},
    )
    return json.loads(r.text)

results = []
N = 100  # in-session scope; set to len(sample) at home
for start in range(0, N, 25):
    chunk = sample.concern_text[start:start+25].tolist()
    results += classify_batch(chunk, start + 1)
    print(f"rows {start+1}-{start+len(chunk)} done")
labels = pd.DataFrame(results).sort_values("n").reset_index(drop=True)
labels["category"].value_counts()

## 3. Validate against a human: you\n\nBefore looking at the model's answers for rows 1–20, code them yourself with only the codebook (pen and paper is fine), then type your labels below. *No peeking — the whole point dies with a peek.*

In [ ]:
my_labels = []  # e.g. ["JOB", "REL", ...]  <- your 20 codes for rows 1-20, in order
if my_labels:
    from sklearn.metrics import cohen_kappa_score, confusion_matrix, accuracy_score
    model_20 = labels.category[:20].tolist()
    print("agreement:", accuracy_score(my_labels, model_20))
    print("Cohen's kappa:", round(cohen_kappa_score(my_labels, model_20), 2))
    cats = sorted(set(my_labels) | set(model_20))
    print(pd.DataFrame(confusion_matrix(my_labels, model_20, labels=cats), index=cats, columns=cats))
else:
    print("Fill in my_labels first — 20 codes, rows 1-20, codebook only.")

## 4. Error analysis — the part that makes it research\n\nFor every disagreement: who was wrong, you or the model? What codebook ambiguity does it expose? (Session discussion; in A4 this becomes your written error table.)\n\n## 5. Robustness probe\n\nRerun rows 1–50 after paraphrasing the codebook instruction line — does the model agree with itself? A classifier that changes its mind when the prompt is reworded is measuring the prompt, not the construct.

In [ ]:
# Take-home: copy classify_batch, reword the instruction sentence (not the codebook),
# rerun rows 1-50, and compare:  (labels_v1 == labels_v2).mean()